In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_pry")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsproyectoyr1")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/sales.csv"

In [0]:
sales_schema = StructType(fields=[
    StructField("SalesID", IntegerType(), False),
    StructField("SalesPersonID", IntegerType(), True),
    StructField("CustomerID", IntegerType(), True),
    StructField("ProductID", IntegerType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("Discount", DecimalType(10,2), True),
    StructField("TotalPrice", DecimalType(10,2), True),
    StructField("SalesDate", TimestampType(), True),
    StructField("TransactionNumber", StringType(), True),
    StructField("ingestion_date", TimestampType(), True)
])

In [0]:
sales_df = spark.read \
            .option("header", True) \
            .schema(sales_schema) \
            .csv(ruta)

In [0]:
sales_final_df = sales_df.withColumn("ingestion_date", current_timestamp())

In [0]:
#sales_final_df.write.mode('overwrite').option('overwriteSchema', 'true').partitionBy('SalesDate').saveAsTable(f'{catalogo}.{esquema}.sales')
sales_final_df.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.sales")